# 63장 · RAG 서비스 함수와 응답 계약

서비스 함수는 검색과 답변 생성을 호출자가 사용할 한 가지 인터페이스로 묶습니다. 근거가 없는 경우에도 일관된 형식으로 결과를 돌려줍니다.

중간 프로젝트의 과제 제출 안내를 서비스 함수로 구성합니다. 한 요청에서 문서 검색, 출처 표시, 사용자별 대화 기록이 연결되는지 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 62장의 결과 열기

같은 폴더에 내려받은 노트북을 여세요. 먼저 my-course/agent-62.json를 읽습니다. 파일이 없으면 시작 셀의 완성 예시로 연습할 수 있습니다.

In [ ]:
from pathlib import Path
import json
folder = Path.cwd()
if (folder.name == "daily" or (folder.name.startswith("chapter") and folder.name[7:].isdigit())):
    folder = folder.parent
work = folder / "my-course"
work.mkdir(exist_ok=True)
source_file = work / "agent-62.json"
if source_file.exists():
    previous = json.loads(source_file.read_text(encoding="utf-8"))
else:
    previous = {'record': {'owner': 'alice', 'id': 'loan-01', 'text': '대출 기간은 14일입니다.'},
     'evidence': {'id': 'loan-01', 'text': '대출 기간은 14일입니다.'},
     'events': [{'name': 'lookup_loan', 'args': {'doc_id': 'loan-01'}}]}
print("내 파일" if source_file.exists() else "완성 예시", previous["record"]["text"])

완성 예시 대출 기간은 14일입니다.


## search()의 입력 문서와 조회 결과

문서 번호와 페이지는 역할이 다릅니다. course-guide-v1:p2는 자료를 구별하는 ID이고 이 문장이 인쇄된 쪽은 1쪽입니다. 답변과 출처를 같은 원문에서 가져오면 다음 단계에서 다시 대조할 수 있습니다.


### 셀 01 · search()의 문서 검색

한 문서만 있는 작은 검색입니다. 저장한 자료를 어떤 질문에 반환할지 확인하세요.


In [1]:
document = {"id": "course-guide-v1:p2", "source": "가상 과정 운영 안내 · 45일 수업용 페이지판 (source-pages.html)",
            "page": 1, "text": "과제는 학습 관리 페이지의 과제함에 제출합니다. 제출 뒤 파일 이름과 등록 시간을 확인합니다."}

def search(question):
    return [document] if "과제" in question else []

print(search("과제는 어디에 내나요?")[0]["text"])

과제는 학습 관리 페이지의 과제함에 제출합니다. 제출 뒤 파일 이름과 등록 시간을 확인합니다.


과제함이라는 답과 제출 후 확인할 두 항목을 모두 원문에서 찾을 수 있습니다. 여기서 쓰는 검색은 키워드 하나만 보는 수업용 함수입니다.


### 문서 ID와 페이지의 표기

<p>문서의 id는 course-guide-v1:p2이고 page 필드의 값은 1입니다. 코드가 출력하는 정수와 사람이 보는 ID 안의 표기를 그대로 구분해서 읽습니다.</p>


## answer()의 상태·답변·출처

정상 실행에도 근거가 없을 수 있습니다. 앱이 읽는 run_status에는 실행 상황을, result.status에는 답변을 만들었는지 적습니다. 소유 대화인지 확인한 뒤에만 검색하고, 빈 결과에서는 생성 횟수를 늘리지 않습니다.


### 셀 02 · answer()의 응답 계약

제공 서비스의 읽기 경로를 줄인 함수입니다. 이 요청의 답과 출처 쪽수를 확인하세요.

소유자가 맞지 않으면 조회를 계속하기 전에 거부합니다. 허용된 소유자에게서 검색이 빈 경우에는 근거 부족 결과로 끝납니다. 문서를 찾은 경우에만 그 원문을 건네 답을 만듭니다. 세 경우의 마지막 줄을 각각 찾아보세요.


In [2]:
owners = {"chat-a": "alice"}
def answer(question, *, owner, conversation):
    if conversation not in owners or owners[conversation] != owner:
        raise PermissionError("conversation_not_owned")
    docs = search(question)
    if not docs:
        return {"run_status": "completed", "result": {"status": "no_evidence", "answer": "자료에서 확인할 수 없습니다.", "sources": [], "generation_calls": 0}}
    doc = docs[0]
    result = {"status": "answered", "answer": f"{doc['text']} [{doc['id']}]",
              "sources": [doc], "generation_calls": 1}
    return {"run_status": "completed", "result": result}

row = answer("과제는 어디에 내나요?", owner="alice", conversation="chat-a")
print(row["result"]["answer"])
print(row["result"]["sources"][0]["page"])

과제는 학습 관리 페이지의 과제함에 제출합니다. 제출 뒤 파일 이름과 등록 시간을 확인합니다. [course-guide-v1:p2]
1


두 번째 줄의 1은 sources[0]['page']에서 읽은 쪽수입니다. 첫 줄의 p2는 자료 안의 문단 식별자입니다.

답변을 펼치는 화면에서는 sources의 id로 인용을 찾고 page로 원문을 엽니다. answer는 검색한 문서 한 건을 그대로 두 필드의 재료로 씁니다. 모델 호출 수나 검색 호출 수를 출력하는 셀은 아닙니다.


#### 근거를 찾지 못한 서비스 응답

<mark>답변 문구만 읽으면 검색이 비었는지 생성에 실패했는지 구분하기 어렵습니다.</mark> 같은 응답에서 status와 출처 목록을 함께 확인하세요. 팀이 화면을 만들 때에도 근거 없는 답 옆에 이전 문서가 붙는 일을 막을 수 있습니다.

선수 개념 · 24장 PDF 챗봇의 요청 추적

#### 소유자 확인을 지나 원문과 인용이 답변이 될 때

```mermaid
flowchart TD
nowner["소유자 확인"]
  nsearch["문서 검색"]
  nhit["근거 선택"]
  nnone["근거 없음"]
  nresult["응답 반환"]
  nowner --> nsearch
  nsearch --> nhit
  nsearch --> nnone
  nhit --> nresult
  nnone --> nresult
```

이 예제의 answer는 로컬 함수입니다. 답변과 generation_calls 필드를 노트북에서 어떻게 만드는지 보세요.

### 처리 결과와 답변 상태를 따로 읽기

<p>run_status는 요청 처리가 끝났는지를, result.status는 문서로 답할 수 있었는지를 나타냅니다. 주차 질문은 처리가 끝나도 no_evidence입니다.</p>


### 검색보다 먼저 확인하는 소유자

<p>answer는 owners에서 대화의 소유자를 조회해 전달받은 owner와 비교합니다. 일치하지 않으면 PermissionError가 발생하므로 search가 그 요청의 자료를 읽지 않습니다.</p>


### 생성 횟수 필드 읽기

<p>이 셀은 문서 한 건이 있으면 f 문자열로 답변을 만들고 generation_calls에 1을 적습니다. 외부 모델의 실제 사용량을 측정한 값은 아닙니다.</p>


### 과제 03 · 근거 부재 시 생성 생략

같은 사용자가 주차비를 물었습니다. 결과 상태와 생성 횟수를 꺼내세요.


In [3]:
# answer에 주차 질문을 전달하고 result를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>missing = answer(&#x27;주차는 무료인가요?&#x27;, owner=&#x27;alice&#x27;, conversation=&#x27;chat-a&#x27;)[&#x27;result&#x27;]
print(missing[&#x27;status&#x27;], missing[&#x27;generation_calls&#x27;])</code></pre>
<p>실행 결과</p><pre><code>no_evidence 0</code></pre>
<p>0은 문서를 찾아 답을 구성한 횟수입니다. 화면에 표시할 상태는 no_evidence이고 요금은 알 수 없습니다.</p>
</details>


### 근거가 없을 때의 분기

<p>주차 질문을 받은 answer가 돌려준 sources와 generation_calls를 먼저 확인하세요.</p>

<h3>소유자 확인 뒤에 나뉘는 검색 경로</h3>

<p>answer는 대화 소유자가 다르면 search를 호출하기 전에 PermissionError를 냅니다. alice가 chat-a에서 과제 질문을 하면 문서 course-guide-v1:p2를 담아 answered를 반환합니다. 같은 대화에 주차 질문을 보내면 빈 검색 결과에서 즉시 no_evidence와 generation_calls=0을 반환합니다.</p>


## 복습과 실습 점검

### 과제 04 · 다른 사용자의 대화에서 검색 막기

bob이 alice 소유의 chat-a로 과제 제출 위치를 물으면 어떤 응답을 받는지 확인하세요. 같은 질문이므로 검색을 시작하면 과제 안내문을 찾을 수 있습니다.


In [ ]:
# bob, chat-a로 answer를 호출하고 PermissionError의 내용을 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>try:
    answer(&quot;과제는 어디에 내나요?&quot;, owner=&quot;bob&quot;, conversation=&quot;chat-a&quot;)
except PermissionError as error:
    print(error)</code></pre>
<p>실행 결과</p><pre><code>conversation_not_owned</code></pre>
<p>소유자 확인은 search보다 앞에 있습니다. 자료가 없는 질문에는 no_evidence가 돌아오지만 다른 사람의 대화에는 result 자체를 반환하지 않습니다. 화면에서도 두 상황을 구분해야 합니다.</p>
</details>


## 필수 근거 검색에 선택 도구 하나 연결하기

학생이 과제 제출 방법과 제출 오류를 문의할 창구를 함께 묻습니다. 제출 방법은 앞에서 쓰던 문서가 알려 주고, 문의 창구는 별도의 도구가 알려 줍니다. 사용자 확인과 필수 검색은 서비스 함수에서, 선택 도구 실행은 create_agent에서 맡습니다.

### 셀 05 · 서로 다른 두 출처를 준비하기

담당 창구를 알려 주는 도구를 하나 만듭니다. 이 셀은 문서 자료와 도구의 반환값을 직접 확인합니다. 다음 셀에서 두 출처를 한 요청에 연결하세요.

In [ ]:
from langchain.agents import create_agent
from langsmith import tracing_context
import os
from langchain_openai import ChatOpenAI

# 수업 커널에 %pip install langchain-openai 를 실행하고 OPENAI_API_KEY를 설정하세요.
model = ChatOpenAI(model=os.getenv("OPENAI_MODEL") or "gpt-4o-mini", timeout=45, max_retries=1)
from langchain_core.tools import tool

@tool
def support_desk() -> str:
    """가상 과정의 제출 오류 문의 창구를 확인합니다."""
    return "운영 지원 창구"
print(document["id"], support_desk.invoke({}))

문서 ID와 문의 창구가 출력됩니다. 제출 장소와 문의 창구를 다른 사실로 취급하면, 어느 답이 틀렸을 때 어떤 자료부터 고쳐야 할지 정할 수 있습니다.

### 문서 근거와 문의 도구가 한 응답에 모이는 경로

```mermaid
flowchart TD
    start["대화 소유자 확인"] --> owner{"일치?"}
    owner -->|아니요| stop["오류로 종료"]
    owner -->|예| docs{"검색 근거가 있는가?"}
    docs -->|없음| empty["자료 없음"]
    docs -->|있음| model["문서를 읽는 모델"]
    model --> choice{"도구 요청?"}
    choice -->|있음| desk["문의 창구 조회"]
    desk --> model
    choice -->|없음| answer["답과 문서 ID 반환"]
```

대화 권한과 필수 검색은 ask_service가 결정합니다. 근거를 넣은 뒤에만 에이전트가 실행되며, 선택 도구를 쓴 경우 그 결과가 다시 모델에 전달됩니다.

### 셀 06 · 근거를 먼저 찾고 선택 도구의 결과 받기

문서 검색과 상담 창구 조회를 구분하세요. 검색 결과의 문서 ID와 모델이 선택한 도구의 실행 기록을 각각 확인하세요.

In [ ]:
def ask_service(question, user_id, chat_id, agent):
    if chat_id not in owners or owners[chat_id] != user_id:
        raise PermissionError("conversation_not_owned")
    docs = search(question)
    if not docs:
        return {"status": "no_evidence", "sources": [], "messages": []}
    prompt = question + "\n문서 근거: " + docs[0]["text"]
    with tracing_context(enabled=False):
        history = agent.invoke({"messages": [("user", prompt)]})["messages"]
    return {"status": "answered", "sources": [docs[0]["id"]], "messages": history}

service_agent = create_agent(model, tools=[support_desk], system_prompt="오류 문의 창구를 물으면 support_desk로 확인하세요.")
response = ask_service("과제 제출 방법과 오류 문의 창구는?", "alice", "chat-a", service_agent)
print(response["sources"])
print([(m.name, m.content) for m in response["messages"] if m.type == "tool"])

['course-guide-v1:p2']
[('support_desk', '운영 지원 창구')]


첫 줄은 검색한 문서의 ID이고 두 번째 줄은 도구 이름과 실제 반환값입니다. 지원 창구라는 문장이 답변에 보이는지만 검사하면 자료의 출처를 구별하기 어렵습니다. 이 예제에서는 문서 ID와 도구 결과를 따로 꺼낼 수 있습니다.

### 과제 07 · 근거가 없을 때도 열 수 있는 응답 화면 구성하기

status가 no_evidence면 “자료 없음”과 빈 출처 목록을 반환하세요. answered면 마지막 메시지 본문과 문서 ID 목록을 반환하는 함수 한 개를 작성하세요. 주차 질문과 앞의 정상 응답에 각각 적용하세요.

In [6]:
parking = ask_service("주차는 어떻게 하나요?", "alice", "chat-a", service_agent)
# def show_answer(row): ...

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>parking = ask_service(&quot;주차는 어떻게 하나요?&quot;, &quot;alice&quot;, &quot;chat-a&quot;, service_agent)
def show_answer(row):
    if row[&quot;status&quot;] == &quot;no_evidence&quot;:
        return &quot;자료 없음&quot;, []
    return row[&quot;messages&quot;][-1].content, row[&quot;sources&quot;]

print(show_answer(parking))
print(show_answer(response))</code></pre>
<p>실행 결과</p><pre><code>(&#x27;자료 없음&#x27;, [])
(&#x27;과제함에 제출하고 오류는 운영 지원 창구에 문의하세요.&#x27;, [&#x27;course-guide-v1:p2&#x27;])</code></pre>
<p>주차 질문은 빈 messages를 돌려주므로 무조건 [-1]을 읽으면 IndexError가 납니다. 자료 없음 분기를 먼저 처리하면 필수 근거가 없을 때도 화면이 정상적으로 이어집니다. 정상 요청에서는 답과 course-guide-v1:p2가 함께 보입니다.</p>
</details>

<mark>서비스가 반드시 수행하는 검색과 에이전트가 선택하는 도구를 각기 다른 기록에서 확인하세요.</mark> 문서 ID는 search에서 만들고, support_desk의 실행 결과는 ToolMessage에 남습니다. 이 로컬 대본은 두 경로가 이어지는지 검사할 때 씁니다.

확인 절차가 필요한 도구를 추가할 때는 데이터를 저장하는 지점을 따로 나누게 됩니다.

## 응답 함수에서 답변과 출처를 함께 반환하기

62장의 도구가 돌려준 원문을 도서관 문의 화면에 보낼 응답으로 바꿉니다. 근거를 못 찾은 경우에도 화면에서 읽을 필드 이름은 같아야 합니다.

In [ ]:
evidence = previous["evidence"]
def desk_answer(question, evidence):
    if "대출" in question and evidence["text"]:
        return {"status": "answered", "answer": evidence["text"], "sources": [evidence["id"]]}
    return {"status": "no_evidence", "answer": "근거를 찾지 못했습니다.", "sources": []}
question = "대출 기간은 며칠인가요?"
response = desk_answer(question, evidence)
print(response["status"], response["sources"])

answered ['loan-01']


답변을 만든 분기는 62장에서 받은 ID를 sources 목록에 넣습니다. 빈 근거로 답할 수 없을 때도 status와 sources가 남으므로 화면이 두 경우를 구분할 수 있습니다.

### 직접 확인하기

대출 규정으로 답할 수 없는 주차 질문을 넣으세요. 보류 응답에 이전 출처가 남는지 확인합니다.

In [ ]:
# 주차 질문을 넣고 status와 sources를 읽으세요.

<details><summary>정답 코드와 실행 결과 보기</summary><pre><code>parking = desk_answer(&quot;주차 요금은 얼마인가요?&quot;, evidence)
print(parking[&quot;status&quot;], parking[&quot;sources&quot;])</code></pre><p>기본 자료로 실행한 결과</p><pre><code>no_evidence []</code></pre><p>새 응답을 만들 때 빈 출처 목록을 주므로 직전 대출 답변의 ID가 섞이지 않습니다.</p></details>

### 64장에 넘길 결과 저장하기

64장은 사용자가 이 답변을 문의 기록으로 남겨도 된다고 승인할 때만 저장합니다. 저장 셀을 다시 실행하면 같은 파일에 이번 결과를 기록합니다.

In [ ]:
product = {"record": previous["record"], "question": question, "response": response, "events": previous["events"]}
target = work / "agent-63.json"
target.write_text(json.dumps(product, ensure_ascii=False, indent=2), encoding="utf-8")
print(target.name)

agent-63.json
